# Alpha Lab: supervised multi-timeframe baseline
This trains a small MLP, **not Kronos fine-tuning or RL**. CPU is sufficient; optionally select a GPU runtime. FP32 is intentional for this reference. Use a fresh runtime.
Locally run build_training_dataset.py and package_colab.py (see TRAINING.md). Upload only your trusted bundle. Test and raw candles are intentionally excluded. A 30-day dataset is a pipeline smoke test, not trading evidence.

In [ ]:
from google.colab import files
from pathlib import Path, PurePosixPath
import hashlib, json, zipfile
uploaded = files.upload()
assert len(uploaded) == 1, 'Upload one trusted colab bundle zip'
archive_path = Path(next(iter(uploaded)))
root = Path('/content/alpha_lab_training')
assert not root.exists(), 'Use a fresh runtime or a new output directory'
with zipfile.ZipFile(archive_path) as archive:
    members = archive.infolist()
    assert sum(m.file_size for m in members) < 2_000_000_000, 'Bundle too large'
    assert len({m.filename for m in members}) == len(members), 'Duplicate archive paths'
    for member in members:
        p = PurePosixPath(member.filename)
        assert not p.is_absolute() and '..' not in p.parts and ':' not in member.filename and chr(92) not in member.filename
        assert (member.external_attr >> 16) & 0o170000 != 0o120000, 'Symlinks forbidden'
    archive.extractall(root)
hashes = json.loads((root / 'bundle-hashes.json').read_text())
actual = {str(p.relative_to(root)) for p in root.rglob('*') if p.is_file()} - {'bundle-hashes.json'}
assert actual == set(hashes), 'Unexpected/missing bundle files'
for name, expected in hashes.items():
    with (root / name).open('rb') as stream:
        assert hashlib.file_digest(stream, 'sha256').hexdigest() == expected, name
assert not (root / 'dataset/test.parquet').exists()
print('Bundle hashes verified. They detect corruption, not an untrusted author.')


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(root / 'requirements.txt')], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '-e', str(root)], check=True)
subprocess.run([sys.executable, '-c', 'import torch; print(torch.__version__, torch.cuda.is_available())'], check=True)


In [ ]:
# Separate process avoids notebook import/ABI conflicts after pip installation.
subprocess.run([sys.executable, 'scripts/train_baseline.py', '--dataset', 'dataset', '--output', 'checkpoint'], cwd=root, check=True)
metadata = json.loads((root / 'checkpoint/metadata.json').read_text())
print(json.dumps({k: metadata[k] for k in ['model_type', 'temperature', 'validation_uncalibrated', 'test_opened']}, indent=2))


In [ ]:
output = Path('/content/alpha_lab_checkpoint.zip')
with zipfile.ZipFile(output, 'x', zipfile.ZIP_DEFLATED) as archive:
    for name in ['model.safetensors', 'metadata.json']:
        archive.write(root / 'checkpoint' / name, name)
files.download(str(output))


Extract your own downloaded checkpoint to a new artifacts/checkpoints directory on the local machine. Run evaluate_checkpoint.py with the **same immutable dataset** and a new report path. Never change the threshold after inspecting this test. No Google Drive credentials, test files, or exchange keys are required. Colab execution must be verified in your runtime; a local smoke pass alone does not prove Colab compatibility.